# Phase 3 CUAD diagnostic on Google Colab (same job as the Kaggle notebook)

**Diagnostic only.** The pre-registered Step A results (all STEP_A_FAIL) are final and untouched. Every output is labelled
CORRECTED_DIAGNOSTIC.

- **Code:** commit `87bd32b46829c9096674d32bb82abaa1b3425156`, verified file by file against its git blob hashes.
- **Models:** roberta-base.
- **Before you start:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all**, and allow Google Drive
  access when asked.
- **Outputs:** saved to `MyDrive/triage_phase3_diag/colab_roberta-base/`.
- **If the session disconnects,** run all again: finished steps and chunks are skipped.

In [ ]:
import base64, hashlib, io, json, os, subprocess, sys, tarfile, time
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
assert out.returncode == 0 and out.stdout.strip(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then Run all"
GPU = out.stdout.strip(); print("GPU:", GPU)
from google.colab import drive
drive.mount("/content/drive")
STATE = "/content/drive/MyDrive/triage_phase3_diag/colab_roberta-base/diag_state"
os.makedirs(STATE, exist_ok=True)

In [ ]:
# embedded code snapshot of commit 87bd32b468, checked against the commit's git blob hashes
TAR_SHA256 = "c81789892ed6371c79bed91f4b5f8431640ad21daf8b64c036bfd16876409785"
BLOBS = {"experiments/followup/comparators/GPU_RUNBOOK.md": "3f6a2cc3604262e15b1e3b959a09172c652cfb85", "experiments/followup/comparators/cuad_qa.py": "c78ac96373dd2a3bab5b1f59e33d48b2f6113457", "experiments/followup/comparators/cuad_questions.json": "f730547cb51dfd4559442db82f863efbd7ca5516", "experiments/followup/comparators/cuad_step_a.py": "a83f31df9ee04f9802a40dc867cba4f27c2e5405", "experiments/followup/comparators/diagnostics/build_diag_notebook.py": "1c355a8d9e960aa0bb3dac599e3d18a34be99c94", "experiments/followup/comparators/diagnostics/contractnli_map_aggregations.py": "3bb2f74b0eb9615fb1644815244b6d3a08fc8670", "experiments/followup/comparators/diagnostics/cuad_compare.py": "9131aa35406186665d7c0ddc79856e76228863a1", "experiments/followup/comparators/diagnostics/cuad_diag_kaggle.py": "61b7c9a686f517ba3ea86fed3cf01febe964fc84", "experiments/followup/comparators/diagnostics/cuad_reference_full.py": "006779cf834506882e9d414c0c12458cef612b0e", "experiments/followup/comparators/diagnostics/cuad_reference_pipeline.py": "8d3e39876f7ee5de21e613fd2fd7748ed2d3f0f8", "experiments/followup/comparators/diagnostics/diag_kaggle.ipynb": "a9065ae1bc9d00f7c3c33f1af34f2482d1769e9e", "experiments/followup/comparators/gpu/EXTERNAL.sha256": "22e58348f6a0f9e67768a53f5422a97fb3bb00bb", "experiments/followup/comparators/gpu/contractnli_step_a.py": "cada1b4b9168116a202a343ac9aa2f5a985c5065", "experiments/followup/comparators/gpu/fetch_external.sh": "73ff29538119497465fe87f7ef6ebe1a688de329", "experiments/followup/comparators/gpu/pair_contractnli.py": "98a7faa9831581e49e7a5cc6bfee29e29528f42d", "experiments/followup/comparators/gpu/run_contractnli_gpu.sh": "9d2e11e0008c39aa061a258edce8aad85bf5cdf2", "experiments/followup/comparators/gpu/run_cuad_gpu.sh": "c5b21c8027e0d19e3bdd9d1e62a491d5155572d7", "experiments/followup/comparators/kaggle/README.md": "efb0f802044f32d0e6b444050e39225207ca69ed", "experiments/followup/comparators/kaggle/SMOKE_CONTRACTNLI.md": "2c784ef03fc8c68c980a88cd1c41b623aeefd441", "experiments/followup/comparators/kaggle/build_notebook.py": "b50d91aaf2e71b0b1bc326899c88b23d149cc956", "experiments/followup/comparators/kaggle/kaggle_remote.py": "1337ac63374dcf37470af8d03f34e199c1656386", "experiments/followup/comparators/kaggle/phase3_driver.py": "5107c233541eaa65c6d71b1dc336b5c0f9048823", "experiments/followup/comparators/kaggle/phase3_kaggle.ipynb": "dcce6d0f2a4f5be43722e3f9ac3d0b96cce94679", "experiments/followup/comparators/kaggle/phase3_report.py": "af8f23fd68cd8f7725f6de654729f05d7951ae15", "experiments/followup/comparators/label_mapping.json": "db795ebd3b0fcd5d6920cd6be09966ea7bc63deb", "experiments/followup/comparators/lexnlp_run.py": "b5ed6cc9a1bb3ced47eb413436f9f1ba811ef62f", "experiments/followup/comparators/lexnlp_sanity.py": "e7d54538334a09f261d03b107161388b3bc7d80c", "experiments/followup/comparators/setup_lexnlp_env.sh": "a19105a6f10e82b8dae62ded18d99a96ad8425ab", "experiments/followup/runs/diagnostics/cuad/sample_seed20261018/titles.txt": "b854433621ca090033b69c71b7366954409f3acc"}
data = base64.b64decode("...")
assert hashlib.sha256(data).hexdigest() == TAR_SHA256, "snapshot corrupted"
REPO = "/content/Triage"
if os.path.exists(REPO):
    subprocess.run(["rm", "-rf", REPO], check=True)
with tarfile.open(fileobj=io.BytesIO(data)) as t:
    t.extractall(REPO)
for path, sha in BLOBS.items():
    b = open(os.path.join(REPO, path), "rb").read()
    assert hashlib.sha1(b"blob %d\0" % len(b) + b).hexdigest() == sha, "file differs from the commit: " + path
COMMIT = open(os.path.join(REPO, "SOURCE_COMMIT")).read().strip()
print("code commit:", COMMIT, "| files verified:", len(BLOBS))

In [ ]:
F = os.path.join(REPO, "experiments/followup")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.6", "tokenizers==0.22.2"], check=True, timeout=1800)
subprocess.run(["bash", "comparators/gpu/fetch_external.sh"], cwd=F, check=True, stdout=subprocess.DEVNULL, timeout=3600)
import torch, transformers
env = {"provider": "colab", "started": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "gpu": GPU, "commit": COMMIT,
       "cuda": torch.cuda.is_available(), "torch": torch.__version__, "transformers": transformers.__version__,
       "python": sys.version.split()[0]}
with open(os.path.join(STATE, "..", "environment.jsonl"), "a") as fh:
    fh.write(json.dumps(env) + "\n")
print(env)
assert env["cuda"], "CUDA not available"

In [ ]:
MODELS, BUDGET = ['roberta-base'], 9.0
log_path = os.path.join(STATE, "..", "notebook_run.log")
import threading
with open(log_path, "a") as fh:
    p = subprocess.Popen([sys.executable, "comparators/diagnostics/cuad_diag_kaggle.py", "--state", STATE,
                          "--tmp", "/content/diag_tmp", "--budget-hours", str(BUDGET), "--models", *MODELS],
                         cwd=F, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    watchdog = threading.Timer((BUDGET + 0.5) * 3600, p.kill)   # hard stop 30 min after the job's own budget
    watchdog.start()
    for line in p.stdout:
        print(line, end=""); fh.write(line); fh.flush()
    p.wait(); watchdog.cancel()
print("exit", p.returncode, "(killed by watchdog; state so far is kept on Drive)" if p.returncode == -9 else "")
lp = os.path.join(STATE, "diag_log.jsonl")
for l in open(lp) if os.path.exists(lp) else []:
    print(l.strip())